In [13]:
import pandas as pd
import numpy as np
from pybdm import BDM
from pybdm import PartitionCorrelated
from statistics import harmonic_mean

In [2]:
exp_path = "C:\\Users\\luano\\Downloads\\RandomBinarySeriesWithLLMs.csv"
bin_seq_df = pd.read_csv(exp_path)

In [4]:
models = ['gpt_4o', 'claude_3.5', 'gpt_4o_mini', 'cursor_small', 'gemini', 'meta', 'o1_mini', 'o1_preview', 'mistral', 'CTM/BDM']

In [6]:
def ascii_to_binary_list(text):
    binary_list = []
    for char in text:
        # Convert each character to its binary representation
        binary_representation = format(ord(char), '08b')
        # Extend the binary_list with the numerical digits of the binary representation
        binary_list.extend([int(bit) for bit in binary_representation])
    return np.array(binary_list).astype(np.int8)

def bdm_compressed(text):
    bdm = BDM(ndim=1,partition=PartitionCorrelated)
    return bdm.bdm(ascii_to_binary_list(text))

In [14]:
def tst_calc(v1,v2):
    v3 = v2[:3]*np.array([1,0.5,0.25])
    return np.sum(v1[:3]*v3)-v1[-1]

test_vals=[]
dd=3
bdm = BDM(ndim=1,partition=PartitionRecursive)
for mdl in models[:-1]:
    sep_df= bin_seq_df[['sequence',f'{mdl}-formula',f'{mdl}-formula-correctness',f'{mdl}-formula-ordinal',f'{mdl}-formula-copy_seq']].copy()
    sep_df["bdm_formula"]=[bdm.bdm(ascii_to_binary_list(x)) for x in sep_df[f'{mdl}-formula'].to_numpy()]
    sep_df["bdm_input"]=[bdm.bdm(ascii_to_binary_list(x)) for x in sep_df['sequence'].to_numpy()]
    sep_df["bdm_min"] = [bdm.bdm(seq) for seq in np.array([np.array(x.split(",")).astype(np.int8) for x in sep_df["sequence"].to_numpy()])]
    df_c_n_n = sep_df[sep_df[f"{mdl}-formula-correctness"] & ~sep_df[f"{mdl}-formula-ordinal"] & ~sep_df[f"{mdl}-formula-copy_seq"]]
    df_c_o = sep_df[sep_df[f"{mdl}-formula-correctness"] & sep_df[f"{mdl}-formula-ordinal"]]
    df_c_p = sep_df[sep_df[f"{mdl}-formula-correctness"] & sep_df[f"{mdl}-formula-copy_seq"]]
    df_i = sep_df[~sep_df[f"{mdl}-formula-correctness"]]
    v1=np.array([len(x)/100 for x in [df_c_n_n,df_c_o,df_c_p,df_i]])
    v2 = []
    for x in [df_c_n_n,df_c_o,df_c_p,df_i]:
        datax = (x["bdm_min"]/x["bdm_formula"]).to_numpy()
        if len(datax)>0:
            v2.append(harmonic_mean(datax))
        else:
            v2.append(0)
    v2 = (np.nan_to_num(np.array(v2)))
    tst = tst_calc(v1,v2)
    test_vals.append([mdl]+list(v1)+list(v2[:3])+[tst])

In [15]:
df_ranking = pd.DataFrame(test_vals)
df_ranking.columns = ["Model","p1","p2","p3","p4","r1","r2","r3","tst"]
df_ranking.sort_values(by=['tst'],ascending=False).set_index(["Model"])

,p1,p2,p3,p4,r1,r2,r3,tst
Model,,,,,,,,
gpt_4o_mini,0.00,0.00,1.0,0.00,0.000000,0.000000,0.141100,0.035275
cursor_small,0.00,0.00,1.0,0.00,0.000000,0.000000,0.141100,0.035275
gemini,0.00,0.00,1.0,0.00,0.000000,0.000000,0.141100,0.035275
mistral,0.00,0.00,1.0,0.00,0.000000,0.000000,0.115663,0.028916
o1_mini,0.00,0.64,0.0,0.36,0.000000,0.058448,0.000000,-0.341297
o1_preview,0.00,0.29,0.0,0.71,0.000000,0.047635,0.000000,-0.703093
claude_3.5,0.06,0.14,0.0,0.80,0.061253,0.047818,0.000000,-0.792978
gpt_4o,0.00,0.00,0.0,1.00,0.000000,0.000000,0.000000,-1.000000
meta,0.00,0.00,0.0,1.00,0.000000,0.000000,0.000000,-1.000000
